In [ ]:
import os, json, time, shutil, hashlib, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
PROFILE = os.environ.get('NEURAL_DNA_PROFILE', 'local')
INPUT, OUTPUT = TASK / 'input' / PROFILE, TASK / 'output' / PROFILE
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
cfg = json.loads((INPUT / 'config.json').read_text())
SEED = cfg['seed']
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(cfg['threads'])

In [ ]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [ ]:
with np.load(INPUT / 'tokens.npz') as data:
    tokens = {key: data[key] for key in data.files}
with np.load(INPUT / 'source_fingerprints.npz') as data:
    source_fingerprints = {key: data[key] for key in data.files}
layout = json.loads((INPUT / 'marker_layout.json').read_text())
device = device_for(os.environ.get('NEURAL_DNA_DEVICE', 'auto'))
model = load_model(INPUT / 'model', device)
base = load_file(str(INPUT / 'model/model.safetensors'))

In [ ]:
values, truth, records = [], [], []
for split, count in [('cal', cfg['calibration_per_kind']), ('test', cfg['test_per_kind'])]:
    for kind, mean in [('long', cfg['mean_long_tract']), ('short', cfg['mean_short_tract'])]:
        for replicate in range(count):
            ancestry = ancestry_path(rng, layout['blocks'], mean)
            splice_blocks(model, base, INPUT / 'donors', split, ancestry)
            initial = fingerprint(model, layout)
            expected = source_fingerprints[f'exact_{split}'][ancestry, 0, np.arange(layout['blocks'])]
            assert np.array_equal(initial, expected)
            optimizer, previous = None, 0
            for steps in cfg['finetune_steps']:
                start = time.perf_counter()
                optimizer = train_steps(model, tokens['training'], steps - previous, rng, cfg, optimizer)
                previous = steps
                observed = fingerprint(model, layout)
                relative_drift = np.sqrt(np.mean((observed - initial) ** 2) / max(np.mean(initial ** 2), 1e-30))
                records.append({'query_id': len(records), 'mosaic_id': f'{split}_{kind}_{replicate:03d}',
                    'split': split, 'kind': kind, 'steps': steps, 'marker_relative_drift': float(relative_drift),
                    'training_seconds': time.perf_counter() - start, **language_metrics(model, tokens['evaluation_ids'])})
                values.append(observed)
                truth.append(ancestry.copy())
            del optimizer
np.savez_compressed(OUTPUT / 'query_fingerprints.npz', values=np.stack(values))
np.savez_compressed(OUTPUT / 'truth.npz', ancestry=np.stack(truth))
pd.DataFrame(records).to_csv(OUTPUT / 'queries.csv', index=False)
del model, base
release_memory()
display(pd.DataFrame(records).groupby(['split', 'kind', 'steps']).agg(
    mosaics=('mosaic_id', 'nunique'), perplexity=('perplexity', 'mean'), marker_relative_drift=('marker_relative_drift', 'mean')))